In [9]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

# STEP 1: SETUP PYSPARK JOB

## Basic Spark Setup

In [ ]:
from pyspark.sql import SparkSession

def create_spark_session():
    return SparkSession.builder \
        .appName("SalaryDetectionJob") \
        .config("spark.python.worker.timeout", "120") \
        .config("spark.executor.heartbeatInterval", "60s") \
        .getOrCreate()

## Entry point

In [15]:
spark = create_spark_session()

df = spark.read.csv(
        "C:/Users/ACER/Downloads/salary-detector/data/test_scenarios.csv",
        # "C:/Users/ACER/Downloads/salary-detector/jobs/salary_dataset_9k.csv",
        header=True,
        inferSchema=True
    )

df.show(5)

KeyboardInterrupt: 

# STEP 2: CLEAN & STANDARDIZE DATA

In [ ]:
from pyspark.sql.functions import col, to_date, when

# df = df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
df = df.withColumn(
    "Date",
    to_date(col("Date"))
)
df = df.withColumn("Amount", col("Amount").cast("double"))
df = df.withColumn("Balance", col("Balance").cast("double"))
print("Before filter:", df.count())
df = df.filter(col("Date").isNotNull())
print("After filter:", df.count())

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, expr, sum as spark_sum

window_txn = Window.partitionBy("CustomerId").orderBy("Date")

df = df.withColumn(
    "prev_balance",
    lag("Balance").over(window_txn)
)

In [ ]:
salary_txn_df = df.filter(
    upper(col("Transaction Details")).rlike("SALARY")
)
post_salary_spend = df.join(
    salary_txn_df.select("CustomerId", "Date").withColumnRenamed("Date", "salary_date"),
    "CustomerId"
).filter(
    (col("Date") > col("salary_date")) &
    (col("Date") <= col("salary_date") + expr("INTERVAL 3 DAYS"))
)

behavior_df = post_salary_spend.groupBy("CustomerId").agg(
    spark_sum(when(col("Type") == "DEBIT", col("Amount")).otherwise(0)).alias("spend_after_salary")
)
behavior_df = behavior_df.join(
    df.groupBy("CustomerId").agg({"Balance": "min"}).withColumnRenamed("min(Balance)", "min_balance"),
    "CustomerId",
    "left"
)

behavior_df = behavior_df.withColumn(
    "quick_drain_flag",
    col("min_balance") < 2000
)

In [ ]:
from pyspark.sql.functions import lag

window_salary = Window.partitionBy("CustomerId").orderBy("Date")

salary_txn_df = df.filter(
    upper(col("Transaction Details")).rlike("SALARY")
)
salary_df = salary_df.withColumn(
    "prev_salary",
    lag("Amount").over(window_salary)
)

salary_df = salary_df.withColumn(
    "salary_drop_flag",
    col("Amount") < 0.7 * col("prev_salary")
)

In [ ]:
salary_drop_df = salary_df.groupBy("CustomerId").agg(
    spark_sum(when(col("salary_drop_flag"), 1).otherwise(0)).alias("salary_drop_count")
)

behavior_df = behavior_df.join(
    salary_drop_df,
    "CustomerId",
    "left"
)

# STEP 3: FILTER CREDIT TRANSACTIONS

## PySpark version:

In [ ]:
from pyspark.sql.functions import upper

credit_df = df.filter(
    (upper(col("Type")).isin("CREDIT", "CR")) &
    (col("Amount") >= 3000)
)

In [ ]:
all_customers_df = df.select("CustomerId").distinct()

# STEP 4: SENDER EXTRACTION

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def extract_sender(details):
    if not details:
        return "UNKNOWN"

    details = details.upper()
    parts = details.split("/")

    if len(parts) >= 2:
        sender = parts[1]
        words = sender.split()

        sender = " ".join(words[:2])

        if sender.replace(" ", "").isdigit():
            return f"ANON_{sender.replace(' ', '')}"

        return sender.strip()

    return "UNKNOWN"

extract_sender_udf = udf(extract_sender, StringType())

In [ ]:
credit_df = credit_df.withColumn(
    "sender",
    extract_sender_udf(col("Transaction Details"))
)

# STEP 5: MERGE SAME-DAY TRANSACTIONS

In [ ]:
from pyspark.sql.functions import sum as spark_sum

daily_df = credit_df.groupBy(
    "CustomerId", "sender", "Date"
).agg(
    spark_sum("Amount").alias("daily_amount")
)

In [ ]:
from pyspark.sql.functions import collect_list

# STEP 6: COMPUTE INTERVALS

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, datediff

window_spec = Window.partitionBy("CustomerId", "sender").orderBy("Date")

daily_df = daily_df.withColumn(
    "prev_date",
    lag("Date").over(window_spec)
)

daily_df = daily_df.withColumn(
    "interval_days",
    datediff(col("Date"), col("prev_date"))
)

# STEP 7: AGGREGATE FEATURES

## Feature aggregation:

In [ ]:
from pyspark.sql.functions import collect_list, avg, stddev, count

features_df = daily_df.groupBy(
    "CustomerId", "sender"
).agg(
    avg("interval_days").alias("interval_mean"),
    stddev("interval_days").alias("interval_std"),
    avg("daily_amount").alias("amount_mean"),
    stddev("daily_amount").alias("amount_std"),
    count("*").alias("count"),
    collect_list("daily_amount").alias("salary_history")  # ✅ THIS
)

In [ ]:
from pyspark.sql.functions import when

features_df = features_df.withColumn(
    "salary_cycle",
    when(col("interval_mean").isNull(), "INSUFFICIENT_DATA")
    .when((col("interval_mean") >= 25) & (col("interval_mean") <= 35), "MONTHLY")
    .when((col("interval_mean") >= 5) & (col("interval_mean") <= 9), "WEEKLY")
    .when((col("interval_mean") >= 12) & (col("interval_mean") <= 18), "BI-WEEKLY")
    .otherwise("IRREGULAR")
)

In [ ]:
from pyspark.sql.functions import col, lower

# 1. periodic check
features_df = features_df.withColumn(
    "is_periodic",
    (
        col("interval_mean").isNotNull() &
        (
            col("interval_mean").between(24, 35) |   # monthly
            col("interval_mean").between(5, 10)  |   # weekly
            col("interval_mean").between(10, 22)     # semi-monthly / flexible
        )
    )
)

# 2. stable amount
features_df = features_df.withColumn(
    "is_stable_amount",
    col("amount_std") < 10000
)

# 3. repetition
features_df = features_df.withColumn(
    "has_repetition",
    col("count") >= 2
)

features_df = features_df.withColumn(
    "has_salary_keyword",
    col("sender").rlike("TCS|INFOSYS|WIPRO|PAYROLL")
)

features_df = features_df.withColumn(
    "fnf_flag",
    (col("amount_mean") > 100000) &
    (col("interval_mean") > 45)
)

In [ ]:
from pyspark.sql.functions import when

features_df = features_df.withColumn(
    "salary_confidence",
    when(col("count") >= 3, "HIGH")
    .when(col("count") == 2, "MEDIUM")
    .otherwise("LOW")
)

In [ ]:
features_df = features_df.withColumn(
    "is_salary_account",
    (
        col("is_periodic") &
        col("is_stable_amount") &
        col("has_repetition") &
        (
            col("has_salary_keyword") |
            (col("amount_mean") > 30000)
        )
    )
)

In [ ]:
from pyspark.sql.functions import sort_array

features_df = features_df.withColumn(
    "salary_history",
    sort_array(col("salary_history"))
)

In [ ]:
from pyspark.sql.functions import coalesce, lit

features_df = features_df.withColumn(
    "interval_std",
    coalesce(col("interval_std"), lit(0.0))
)

features_df = features_df.withColumn(
    "amount_std",
    coalesce(col("amount_std"), lit(0.0))
)

features_df = features_df.withColumn(
    "amount_mean",
    coalesce(col("amount_mean"), lit(1.0))  # avoid division issues
)

# STEP 8: APPLY SALARY RULES

In [ ]:
features_df = features_df.withColumn(
    "is_periodic",
    (
        col("interval_mean").isNotNull() &
        (
            col("interval_mean").between(24, 35) |   # monthly
            col("interval_mean").between(5, 10)  |   # weekly
            col("interval_mean").between(10, 22)     # semi-monthly / flexible
        )
    )
)

features_df = features_df.withColumn(
    "is_stable_amount",
    col("amount_std") < 10000
)

features_df = features_df.withColumn(
    "has_repetition",
    col("count") >= 2
)


In [ ]:
df.filter(col("CustomerId") == "C4").show(truncate=False)

# STEP 9: SCORING

## Score calculation:

In [ ]:
features_df = features_df.withColumn(
    "time_score",
    1 / (1 + col("interval_std"))
)

features_df = features_df.withColumn(
    "amount_score",
    1 / (1 + (col("amount_std") / (col("amount_mean") + 1)))
)

features_df = features_df.withColumn(
    "salary_boost",
    (col("amount_mean") > 30000).cast("int")
)

features_df = features_df.withColumn(
    "count_score",
    (col("count") / 5)
)

features_df = features_df.withColumn(
    "final_score",
    (
        0.5 * col("time_score") +
        0.25 * col("amount_score") +
        0.15 * col("count_score") +
        0.1 * col("salary_boost")
    )
)

# STEP 10: PICK BEST SENDER PER CUSTOMER

## Window ranking

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

rank_window = Window.partitionBy("CustomerId").orderBy(col("final_score").desc())

ranked_df = features_df.withColumn(
    "rank",
    row_number().over(rank_window)
)

best_df = ranked_df.filter(col("rank") == 1)

## STEP 11: OUTPUT

In [ ]:
features_df.select(
    "CustomerId",
    "sender",
    "salary_cycle",
    "count",
    "interval_mean",
    "amount_mean",
    "is_periodic",
    "is_stable_amount",
    "has_repetition",
    "has_salary_keyword",
    "salary_confidence"
).show(truncate=False)

In [ ]:
# attach behavior FIRST
best_df = best_df.join(
    behavior_df,
    on="CustomerId",
    how="left"
)

# THEN attach all customers
final_df = all_customers_df.join(
    best_df,
    on="CustomerId",
    how="left"
)

In [ ]:
from pyspark.sql.functions import col, when, lit, concat_ws, format_number

final_output = final_df.select(
    col("CustomerId"),

    # sender
    when(col("sender").isNull(), "NO_SALARY_DETECTED")
    .otherwise(col("sender"))
    .alias("sender"),

    # salary history (array → string)
    when(col("salary_history").isNull(), "-")
    .otherwise(concat_ws(", ", col("salary_history")))
    .alias("salary_history"),

    # avg salary (numeric safe)
    when(col("amount_mean").isNull(), "-")
    .otherwise(format_number(col("amount_mean"), 0))
    .alias("avg_salary"),

    # cycle
    when(col("salary_cycle").isNull(), "NO_PATTERN")
    .otherwise(col("salary_cycle"))
    .alias("salary_cycle"),

    # score
    when(col("final_score").isNull(), "0.00")
    .otherwise(format_number(col("final_score"), 2))
    .alias("score"),

    # confidence
    when(col("salary_confidence").isNull(), "NONE")
    .otherwise(col("salary_confidence"))
    .alias("salary_confidence"),

    # flag (boolean stays boolean)
    when(col("is_salary_account").isNull(), False)
    .otherwise(col("is_salary_account"))
    .alias("is_salary_account"),

    # reason (boolean logic works properly now)
    when(col("sender").isNull(), "NO SALARY SIGNAL")
    .otherwise(
        concat_ws(" | ",
            when(col("is_periodic"), "Periodic").otherwise("Not Periodic"),
            when(col("is_stable_amount"), "Stable Amount").otherwise("Variable"),
            when(col("has_repetition"), "Repeated").otherwise("Not Repeated"),

            when(col("spend_after_salary").isNotNull(),
                concat_ws("", lit("Spend:"), col("spend_after_salary").cast("string"))
            ),

            when(col("quick_drain_flag") == True, "QuickDrain").otherwise("StableBalance"),

            when(col("salary_drop_count") > 0, "SalaryDrop").otherwise("NoDrop"),

            when(col("fnf_flag") == True, "FNF").otherwise("Regular"),

            when(col("has_salary_keyword"), "Keyword").otherwise("No Keyword")
        )
    ).alias("reason")
)
final_output = final_output.orderBy("CustomerId")
final_output.show(truncate=False)